In [ ]:
"""
Cross-dataset generalization test for the five deepfake detectors.

Run AFTER:
  1) your main evaluation script (it writes overall_metrics_comparison.csv, which holds the
     per-model decision thresholds tuned on the in-distribution VALIDATION split), and
  2) prepare_genimage_subset.py (it writes genimage_subset/real and genimage_subset/fake/<generator>).

    python cross_dataset_eval.py --data_dir /kaggle/working/genimage_subset

Protocol (this is what makes the claim defensible):
  * Models are NOT retrained or fine-tuned on the external data.
  * The decision threshold is FIXED to the one chosen on your in-distribution validation split.
    It is never tuned on the external data. (An "oracle" threshold is reported separately,
    for diagnosis only: if AUC stays high but accuracy collapses, the features transfer and
    only the calibration shifted.)
  * Threshold-free metrics (ROC-AUC, AP) are the primary numbers.
  * Paired bootstrap CIs on the AUC difference (dynamic vs every other model) tell you
    whether the adaptive advantage survives outside the training distribution.

Outputs (in --out_dir):
  cross_dataset_overall.csv             all models, external set, fixed source threshold
  cross_dataset_generalization_gap.csv  in-distribution vs external, per model
  cross_dataset_per_generator_long.csv  / _accuracy_wide / _auc_wide / _ap_wide / _detection_rate_wide
  cross_dataset_bootstrap_auc.csv       AUC with 95% CI + paired differences vs dynamic_freqnet
  cross_dataset_composition.csv         file-format / image-size mix per class and generator
  cross_dataset_predictions_long.csv    every prediction
"""
from __future__ import annotations

import argparse
import math
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFile
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True

# -----------------------------------------------------------------------------
# 0. Configuration
# -----------------------------------------------------------------------------

parser = argparse.ArgumentParser()
parser.add_argument("--data_dir", type=Path,
                    default=Path("/kaggle/input/datasets/nocturnalnerd/new-gen-image-subset-dataset"))
parser.add_argument("--out_dir", type=Path, default=Path("/kaggle/working/cross_dataset_evaluation"))
parser.add_argument(
    "--source_metrics_csv",
    type=Path,
    default=Path("/kaggle/working/five_model_evaluation/overall_metrics_comparison.csv"),
    help="overall_metrics_comparison.csv from the in-distribution run (gives thresholds + in-dist metrics).",
)
parser.add_argument("--batch_size", type=int, default=16)
parser.add_argument("--num_workers", type=int, default=8)
parser.add_argument("--n_boot", type=int, default=1000)
parser.add_argument("--seed", type=int, default=42)
args = parser.parse_known_args()[0]  # parse_known_args: harmless inside notebooks

SEED = args.seed
IMAGE_SIZE = 256  # must match the in-distribution evaluation
N_THRESHOLD_CANDIDATES = 181
REFERENCE_MODEL = "dynamic_freqnet"  # paired comparisons are against this model

ORIGINAL_FREQNET_CHECKPOINT_PATH = Path("/kaggle/input/datasets/mikekomari/models/best_freqnet_rgb_baseline (1).pt")
SIMPLE_CNN_CHECKPOINT_PATH = Path("/kaggle/input/datasets/mikekomari/models/normalCNN.pt")
ALEXNET_CHECKPOINT_PATH = Path("/kaggle/input/datasets/mikekomari/models/best_alexnet_scratch.pt")
DYNAMIC_CHECKPOINT_PATH = Path("/kaggle/input/datasets/nocturnalnerd/best-dynamic-freqnet/best_freqnet_dynamic_c0.20_seed42 (2).pt")
RING_ONES_CHECKPOINT_PATH = Path("/kaggle/input/datasets/nocturnalnerd/best-fixed-model-pt/best_freqnet_ring_ones_c0.20_seed42.pt")

REPO_DIR = Path("/kaggle/working/FreqNet-DeepfakeDetection")
REPO_URL = "https://github.com/chuangchuangtan/FreqNet-DeepfakeDetection.git"

args.out_dir.mkdir(parents=True, exist_ok=True)
OUT = args.out_dir

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)
print("Device:", device)

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
freqnet_source = REPO_DIR / "networks" / "freqnet.py"
source_text = freqnet_source.read_text(encoding="utf-8")
patched_text = source_text.replace(".cuda()", "")
if patched_text != source_text:
    freqnet_source.write_text(patched_text, encoding="utf-8")
sys.path.insert(0, str(REPO_DIR))
from networks.freqnet import FreqNet  # noqa: E402

# -----------------------------------------------------------------------------
# 1. External dataset: genimage_subset/real/*  and  genimage_subset/fake/<generator>/*
# -----------------------------------------------------------------------------

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}


def find_subset_root(root: Path) -> Path:
    """Return the folder that directly contains real/ and fake/. Kaggle often adds an extra
    nesting level (e.g. <dataset>/genimage_subset/real), so search up to 3 levels down."""
    if not root.is_dir():
        raise FileNotFoundError(f"{root} does not exist. Check the dataset is attached to this notebook.")
    for dirpath, dirnames, _ in os.walk(root):
        if "real" in dirnames and "fake" in dirnames:
            return Path(dirpath)
        if len(Path(dirpath).relative_to(root).parts) >= 3:
            dirnames[:] = []  # do not descend further
    top = sorted(p.name for p in root.iterdir())[:20]
    raise FileNotFoundError(
        f"Could not find a folder containing both real/ and fake/<generator>/ under {root}. "
        f"Top-level entries: {top}"
    )


def scan_subset(root: Path) -> pd.DataFrame:
    root = find_subset_root(root)
    print("Using external dataset folder:", root)
    rows = []
    for p in sorted((root / "real").rglob("*")):
        if p.suffix.lower() in IMG_EXTS:
            rows.append((str(p), 0, "real"))
    for gen_dir in sorted(d for d in (root / "fake").iterdir() if d.is_dir()):
        for p in sorted(gen_dir.rglob("*")):
            if p.suffix.lower() in IMG_EXTS:
                rows.append((str(p), 1, gen_dir.name))
    df = pd.DataFrame(rows, columns=["path", "label", "generator"])
    if df.empty or df["label"].nunique() < 2:
        raise ValueError("Need both real and fake images in the external subset.")

    fmts, widths, heights = [], [], []
    for p in tqdm(df["path"], desc="reading image headers", leave=False):
        with Image.open(p) as im:  # header only, cheap
            fmts.append((im.format or "UNK").lower())
            widths.append(im.size[0])
            heights.append(im.size[1])
    df["format"], df["width"], df["height"] = fmts, widths, heights
    return df


class FolderDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform):
        self.paths = df["path"].tolist()
        self.labels = df["label"].astype(np.float32).tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(image), torch.tensor(self.labels[idx], dtype=torch.float32)


# -----------------------------------------------------------------------------
# 2. Model classes -- identical to the in-distribution evaluation script
# -----------------------------------------------------------------------------

MID_HIGH_CUTOFF = 0.20
NUM_BANDS = 8
SELECTOR_HIDDEN = 32


class AmpSafeFreqNet(FreqNet):
    def hfreqWH(self, x: torch.Tensor, scale: int) -> torch.Tensor:
        assert scale > 2
        x = x.float()
        x = torch.fft.fft2(x, norm="ortho")
        x = torch.fft.fftshift(x, dim=[-2, -1])
        _, _, h, w = x.shape
        mask = torch.ones((h, w), device=x.device, dtype=x.real.dtype)
        mask[h // 2 - h // scale : h // 2 + h // scale, w // 2 - w // scale : w // 2 + w // scale] = 0.0
        x = x * mask[None, None, :, :]
        x = torch.fft.ifftshift(x, dim=[-2, -1])
        x = torch.fft.ifft2(x, norm="ortho").real
        return F.relu(x, inplace=False)

    def hfreqC(self, x: torch.Tensor, scale: int) -> torch.Tensor:
        assert scale > 2
        x = x.float()
        x = torch.fft.fft(x, dim=1, norm="ortho")
        x = torch.fft.fftshift(x, dim=1)
        _, c, _, _ = x.shape
        mask = torch.ones(c, device=x.device, dtype=x.real.dtype)
        mask[c // 2 - c // scale : c // 2 + c // scale] = 0.0
        x = x * mask[None, :, None, None]
        x = torch.fft.ifftshift(x, dim=1)
        x = torch.fft.ifft(x, dim=1, norm="ortho").real
        return F.relu(x, inplace=False)

    def _frequency_conv(self, x: torch.Tensor, real_conv: nn.Module, imag_conv: nn.Module) -> torch.Tensor:
        x = x.float()
        x = torch.fft.fft2(x, norm="ortho")
        x = torch.fft.fftshift(x, dim=[-2, -1])
        x = torch.complex(real_conv(x.real), imag_conv(x.imag))
        x = torch.fft.ifftshift(x, dim=[-2, -1])
        x = torch.fft.ifft2(x, norm="ortho").real
        return F.relu(x, inplace=False)

    def _raw_projection(self, x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor, stride: int) -> torch.Tensor:
        x = F.conv2d(x.float(), weight.float(), bias.float(), stride=stride, padding=0)
        return F.relu(x, inplace=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.hfreqWH(x, 4)
        x = self._raw_projection(x, self.weight1, self.bias1, stride=1)
        x = self.hfreqC(x, 4)
        x = self._frequency_conv(x, self.realconv1, self.imagconv1)

        x = self.hfreqWH(x, 4)
        x = self._raw_projection(x, self.weight2, self.bias2, stride=2)
        x = self.hfreqC(x, 4)
        x = self._frequency_conv(x, self.realconv2, self.imagconv2)

        x = self.maxpool(x)
        x = self.layer1(x)

        x = self.hfreqWH(x, 4)
        x = self._raw_projection(x, self.weight3, self.bias3, stride=1)
        x = self._frequency_conv(x, self.realconv3, self.imagconv3)

        x = self.hfreqWH(x, 4)
        x = self._raw_projection(x, self.weight4, self.bias4, stride=2)
        x = self._frequency_conv(x, self.realconv4, self.imagconv4)

        x = self.layer2(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        return self.fc1(x)


class DynamicAmpSafeFreqNet(AmpSafeFreqNet):
    MODES = ("dynamic", "static", "ones", "freqnet")

    def __init__(self, first_stage_mode: str = "dynamic", cutoff: float = 0.20, num_bands: int = 8,
                 hidden_dim: int = 32, image_size: int = 256):
        super().__init__()
        if first_stage_mode not in self.MODES:
            raise ValueError(f"first_stage_mode must be one of {self.MODES}")
        self.first_stage_mode = first_stage_mode
        self.cutoff = float(cutoff)
        self.num_bands = int(num_bands)
        self._stem_pending = False

        if first_stage_mode == "dynamic":
            self.dynamic_selector = nn.Sequential(
                nn.Linear(num_bands, hidden_dim), nn.ReLU(inplace=True),
                nn.Linear(hidden_dim, num_bands), nn.Sigmoid(),
            )
        elif first_stage_mode == "static":
            self.static_band_logits = nn.Parameter(torch.zeros(num_bands))

        band_plus1, membership, counts = self._build_band_geometry(image_size, image_size)
        self.register_buffer("_band_plus1", band_plus1, persistent=False)
        self.register_buffer("_membership", membership, persistent=False)
        self.register_buffer("_band_counts", counts, persistent=False)

    def _build_band_geometry(self, h: int, w: int):
        yy = torch.linspace(-1.0, 1.0, h)
        xx = torch.linspace(-1.0, 1.0, w)
        grid_y, grid_x = torch.meshgrid(yy, xx, indexing="ij")
        rho = torch.sqrt(grid_x.square() + grid_y.square())
        rho = rho / rho.max()
        edges = torch.linspace(self.cutoff, 1.0, self.num_bands + 1)
        band = torch.bucketize(rho, edges[1:-1], right=True)
        band = torch.where(rho >= self.cutoff, band, torch.full_like(band, -1))
        band_plus1 = (band + 1).flatten()
        membership = F.one_hot(band_plus1, self.num_bands + 1)[:, 1:].float()
        counts = membership.sum(dim=0)
        if bool((counts == 0).any()):
            raise ValueError("At least one radial band is empty; lower NUM_BANDS or the cutoff.")
        return band_plus1, membership, counts

    def _band_descriptor(self, spectrum: torch.Tensor) -> torch.Tensor:
        h, w = spectrum.shape[-2:]
        log_mag = torch.log1p(spectrum.abs() * math.sqrt(h * w)).mean(dim=1)
        per_band = log_mag.flatten(1) @ self._membership / self._band_counts
        return per_band / (per_band.mean(dim=1, keepdim=True) + 1e-6)

    def _band_weights(self, spectrum: torch.Tensor) -> torch.Tensor:
        batch = spectrum.shape[0]
        if self.first_stage_mode == "dynamic":
            return self.dynamic_selector(self._band_descriptor(spectrum))
        if self.first_stage_mode == "static":
            return torch.sigmoid(self.static_band_logits).unsqueeze(0).expand(batch, -1)
        return torch.ones(batch, self.num_bands, device=spectrum.device, dtype=spectrum.real.dtype)

    def _band_mask(self, weights: torch.Tensor, h: int, w: int) -> torch.Tensor:
        padded = torch.cat([weights.new_zeros(weights.shape[0], 1), weights], dim=1)
        return padded[:, self._band_plus1].reshape(-1, 1, h, w)

    @torch.no_grad()
    def inspect_stem(self, x: torch.Tensor):
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            spectrum = torch.fft.fftshift(torch.fft.fft2(x.float(), norm="ortho"), dim=[-2, -1])
            return self._band_descriptor(spectrum), self._band_weights(spectrum)

    def _first_stage(self, x: torch.Tensor, scale: int) -> torch.Tensor:
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            x = torch.fft.fft2(x, norm="ortho")
            x = torch.fft.fftshift(x, dim=[-2, -1])
            _, _, h, w = x.shape
            if self.first_stage_mode == "freqnet":
                mask = torch.ones((h, w), device=x.device, dtype=x.real.dtype)
                mask[h // 2 - h // scale : h // 2 + h // scale, w // 2 - w // scale : w // 2 + w // scale] = 0.0
                mask = mask[None, None, :, :]
            else:
                mask = self._band_mask(self._band_weights(x), h, w)
            x = x * mask
            x = torch.fft.ifftshift(x, dim=[-2, -1])
            x = torch.fft.ifft2(x, norm="ortho").real
            return F.relu(x, inplace=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self._stem_pending = True
        return super().forward(x)

    def hfreqWH(self, x: torch.Tensor, scale: int) -> torch.Tensor:
        if self._stem_pending:
            self._stem_pending = False
            if x.shape[1] != 3:
                raise RuntimeError(f"Expected the 3-channel image at the first stage; got {tuple(x.shape)}")
            return self._first_stage(x, scale)
        return super().hfreqWH(x, scale)


FIRST_STAGE_FOR_MODE = {"dynamic": "dynamic", "static_ring": "static", "ring_ones": "ones"}


class ExperimentModel(nn.Module):
    def __init__(self, input_mode: str):
        super().__init__()
        if input_mode != "rgb_baseline" and input_mode not in FIRST_STAGE_FOR_MODE:
            raise ValueError(f"input_mode must be 'rgb_baseline' or one of {sorted(FIRST_STAGE_FOR_MODE)}")
        self.input_mode = input_mode
        if input_mode == "rgb_baseline":
            self.backbone = AmpSafeFreqNet()
        else:
            self.backbone = DynamicAmpSafeFreqNet(
                first_stage_mode=FIRST_STAGE_FOR_MODE[input_mode], cutoff=MID_HIGH_CUTOFF,
                num_bands=NUM_BANDS, hidden_dim=SELECTOR_HIDDEN, image_size=IMAGE_SIZE,
            )
        self.register_buffer("imagenet_mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("imagenet_std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, rgb: torch.Tensor) -> torch.Tensor:
        model_input = (rgb - self.imagenet_mean) / self.imagenet_std
        return self.backbone(model_input).flatten()


class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        def block(in_ch: int, out_ch: int, pool: bool = True) -> nn.Sequential:
            layers = [nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
            if pool:
                layers.append(nn.MaxPool2d(2))
            return nn.Sequential(*layers)

        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256), block(256, 256, pool=False))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(256, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x).flatten()


class AlexNetScratch(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=2), nn.BatchNorm2d(96), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(96, 256, kernel_size=5, padding=2), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(256, 384, kernel_size=3, padding=1), nn.BatchNorm2d(384), nn.ReLU(inplace=True),
            nn.Conv2d(384, 384, kernel_size=3, padding=1), nn.BatchNorm2d(384), nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.pool = nn.AdaptiveAvgPool2d((6, 6))
        self.classifier = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(256 * 6 * 6, 4096), nn.ReLU(inplace=True),
            nn.Dropout(0.5), nn.Linear(4096, 4096), nn.ReLU(inplace=True),
            nn.Linear(4096, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x).flatten()


MODEL_KINDS = ["original_freqnet", "simple_cnn", "alexnet_scratch", "ring_ones_freqnet", "dynamic_freqnet"]
CHECKPOINTS = {
    "original_freqnet": ORIGINAL_FREQNET_CHECKPOINT_PATH,
    "simple_cnn": SIMPLE_CNN_CHECKPOINT_PATH,
    "alexnet_scratch": ALEXNET_CHECKPOINT_PATH,
    "ring_ones_freqnet": RING_ONES_CHECKPOINT_PATH,
    "dynamic_freqnet": DYNAMIC_CHECKPOINT_PATH,
}


def build_model(kind: str) -> nn.Module:
    if kind == "original_freqnet":
        return ExperimentModel(input_mode="rgb_baseline")
    if kind == "simple_cnn":
        return SimpleCNN()
    if kind == "alexnet_scratch":
        return AlexNetScratch()
    if kind == "ring_ones_freqnet":
        return ExperimentModel(input_mode="ring_ones")
    if kind == "dynamic_freqnet":
        return ExperimentModel(input_mode="dynamic")
    raise ValueError(f"Unknown model kind: {kind!r}")


def load_checkpoint_into(model: nn.Module, path: Path) -> nn.Module:
    if not path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    state_dict = checkpoint.get("model_state_dict", checkpoint) if isinstance(checkpoint, dict) else checkpoint
    state_dict = {k[len("module.") :] if k.startswith("module.") else k: v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=True)
    return model


# -----------------------------------------------------------------------------
# 3. Inference + metrics
# -----------------------------------------------------------------------------


@torch.inference_mode()
def run_inference(model: nn.Module, kind: str, loader: DataLoader) -> np.ndarray:
    model.eval()
    probs = []
    needs_manual_normalize = kind in {"simple_cnn", "alexnet_scratch"}
    for images, _ in tqdm(loader, desc=f"infer[{kind}]", leave=False):
        images = images.to(device, non_blocking=True)
        if needs_manual_normalize:
            images = (images - IMAGENET_MEAN) / IMAGENET_STD
        probs.append(torch.sigmoid(model(images)).cpu().numpy())
    return np.concatenate(probs)


def best_f1_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    best_t, best_f1 = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, N_THRESHOLD_CANDIDATES):
        f1 = f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_t


def overall_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    oracle_t = best_f1_threshold(y_true, y_prob)
    oracle_pred = (y_prob >= oracle_t).astype(int)
    return {
        "n": int(len(y_true)),
        "threshold_source": threshold,
        "roc_auc": roc_auc_score(y_true, y_prob),
        "average_precision": average_precision_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_fake": precision_score(y_true, y_pred, zero_division=0),
        "recall_fake": recall_score(y_true, y_pred, zero_division=0),
        "specificity_real": tn / max(tn + fp, 1),
        "f1_fake": f1_score(y_true, y_pred, zero_division=0),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        # diagnostic only: threshold tuned ON the external data (not a fair result)
        "oracle_threshold": oracle_t,
        "oracle_accuracy": accuracy_score(y_true, oracle_pred),
        "oracle_f1_fake": f1_score(y_true, oracle_pred, zero_division=0),
    }


def wilson_ci(k: int, n: int, z: float = 1.96):
    if n == 0:
        return (np.nan, np.nan, np.nan)
    p_hat = k / n
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denom
    half = (z * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2))) / denom
    return p_hat, max(0.0, center - half), min(1.0, center + half)


def per_generator_table(y_true, y_prob, generator, threshold) -> pd.DataFrame:
    y_pred = (y_prob >= threshold).astype(int)
    real_mask = y_true == 0
    rows = []
    for gen in sorted(set(generator[y_true == 1])):
        fake_mask = (y_true == 1) & (generator == gen)
        comb = real_mask | fake_mask
        n_gen = int(fake_mask.sum())
        k_gen = int((y_pred[fake_mask] == 1).sum())
        rate, lo, hi = wilson_ci(k_gen, n_gen)
        rows.append({
            "generator": gen,
            "n_fake_images": n_gen,
            "auc_with_reals": roc_auc_score(y_true[comb], y_prob[comb]),
            "average_precision_with_reals": average_precision_score(y_true[comb], y_prob[comb]),
            "accuracy_with_reals": accuracy_score(y_true[comb], y_pred[comb]),
            "detection_rate": rate, "detection_rate_ci_lo": lo, "detection_rate_ci_hi": hi,
        })
    return pd.DataFrame(rows)


def paired_bootstrap_auc(y_true: np.ndarray, probs: dict, n_boot: int, seed: int, reference: str) -> pd.DataFrame:
    """Same resampled images for every model, so differences are paired."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    draws = {k: [] for k in probs}
    for _ in tqdm(range(n_boot), desc="bootstrap", leave=False):
        idx = rng.integers(0, n, n)
        if y_true[idx].min() == y_true[idx].max():
            continue
        for k, p in probs.items():
            draws[k].append(roc_auc_score(y_true[idx], p[idx]))
    draws = {k: np.array(v) for k, v in draws.items()}

    rows = []
    for k, d in draws.items():
        row = {"model": k, "auc": roc_auc_score(y_true, probs[k]),
               "auc_ci_lo": np.percentile(d, 2.5), "auc_ci_hi": np.percentile(d, 97.5)}
        if reference in draws and k != reference:
            diff = draws[reference] - d  # positive => reference better
            row.update({
                f"auc_diff_{reference}_minus_model": float(np.mean(diff)),
                "diff_ci_lo": np.percentile(diff, 2.5),
                "diff_ci_hi": np.percentile(diff, 97.5),
                "p_reference_not_better": float((diff <= 0).mean()),
            })
        rows.append(row)
    return pd.DataFrame(rows)


# -----------------------------------------------------------------------------
# 4. Run
# -----------------------------------------------------------------------------

ext_df = scan_subset(args.data_dir)
print(f"\nExternal set: {len(ext_df):,} images  (real={int((ext_df.label == 0).sum())}, fake={int((ext_df.label == 1).sum())})")
print(ext_df.groupby("generator").size().to_string())

# Shortcut check: does file format or resolution separate real from fake?
composition = (
    ext_df.assign(size=ext_df["width"].astype(str) + "x" + ext_df["height"].astype(str))
    .groupby(["generator", "format"]).agg(n=("path", "size"),
                                          median_width=("width", "median"),
                                          median_height=("height", "median")).reset_index()
)
composition.to_csv(OUT / "cross_dataset_composition.csv", index=False)
print("\nFile-format / size mix per generator (check for a real-vs-fake shortcut):")
print(composition.to_string(index=False))

eval_transform = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True), transforms.ToTensor()])
loader = DataLoader(FolderDataset(ext_df, eval_transform), batch_size=args.batch_size, shuffle=False,
                    num_workers=args.num_workers, pin_memory=(device.type == "cuda"))
y_true = ext_df["label"].to_numpy().astype(np.int64)
generator = ext_df["generator"].to_numpy()

# thresholds + in-distribution reference numbers from the first script
source_df = None
thresholds: dict = {}
if args.source_metrics_csv.exists():
    source_df = pd.read_csv(args.source_metrics_csv).set_index("model")
    thresholds = source_df["threshold"].to_dict()
    print("\nUsing in-distribution validation thresholds:", {k: round(v, 3) for k, v in thresholds.items()})
else:
    print(f"\nWARNING: {args.source_metrics_csv} not found. Falling back to threshold 0.5 for every model; "
          "this is NOT the validation-tuned threshold, so accuracy/F1 are not comparable to the in-distribution run.")

overall_rows, gen_frames, pred_frames = [], [], []
probs_by_model: dict = {}

for kind in MODEL_KINDS:
    print(f"\n{'=' * 70}\n{kind}\n{'=' * 70}")
    try:
        model = load_checkpoint_into(build_model(kind).to(device), CHECKPOINTS[kind])
    except FileNotFoundError as e:
        print(f"SKIPPING {kind}: {e}")
        continue

    prob = run_inference(model, kind, loader)
    probs_by_model[kind] = prob
    thr = float(thresholds.get(kind, 0.5))

    m = overall_metrics(y_true, prob, thr)
    m["model"] = kind
    overall_rows.append(m)
    print(f"{kind}: AUC={m['roc_auc']:.4f}  AP={m['average_precision']:.4f}  "
          f"Acc@src-thr={m['accuracy']:.4f}  F1={m['f1_fake']:.4f}  (oracle Acc={m['oracle_accuracy']:.4f})")

    g = per_generator_table(y_true, prob, generator, thr)
    g["model"] = kind
    gen_frames.append(g)

    pred_frames.append(pd.DataFrame({"model": kind, "y_true": y_true, "y_prob": prob,
                                     "generator": generator, "path": ext_df["path"].to_numpy(),
                                     "format": ext_df["format"].to_numpy()}))
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

if not overall_rows:
    raise SystemExit("No model could be evaluated; check checkpoint paths.")

# -----------------------------------------------------------------------------
# 5. Save tables
# -----------------------------------------------------------------------------

overall = pd.DataFrame(overall_rows)
overall = overall[["model"] + [c for c in overall.columns if c != "model"]]
overall.to_csv(OUT / "cross_dataset_overall.csv", index=False)

gen_long = pd.concat(gen_frames, ignore_index=True)
gen_long.to_csv(OUT / "cross_dataset_per_generator_long.csv", index=False)
counts = gen_long.groupby("generator")["n_fake_images"].first()
present = [k for k in MODEL_KINDS if k in probs_by_model]
for col, name in [("accuracy_with_reals", "accuracy"), ("auc_with_reals", "auc"),
                  ("average_precision_with_reals", "ap"), ("detection_rate", "detection_rate")]:
    wide = gen_long.pivot(index="generator", columns="model", values=col)[present]
    wide.insert(0, "n_test_fakes", counts)
    wide.to_csv(OUT / f"cross_dataset_per_generator_{name}_wide.csv")

pd.concat(pred_frames, ignore_index=True).to_csv(OUT / "cross_dataset_predictions_long.csv", index=False)

# generalization gap: in-distribution test vs external
if source_df is not None:
    gap_rows = []
    for _, r in overall.iterrows():
        k = r["model"]
        if k not in source_df.index:
            continue
        s = source_df.loc[k]
        row = {"model": k}
        for metric in ["roc_auc", "average_precision", "accuracy", "balanced_accuracy", "f1_fake"]:
            row[f"{metric}_in_dist"] = s[metric]
            row[f"{metric}_external"] = r[metric]
            row[f"{metric}_drop"] = s[metric] - r[metric]
        gap_rows.append(row)
    gap = pd.DataFrame(gap_rows)
    gap.to_csv(OUT / "cross_dataset_generalization_gap.csv", index=False)

print("\nBootstrapping AUC CIs and paired differences...")
boot = paired_bootstrap_auc(y_true, probs_by_model, args.n_boot, SEED, REFERENCE_MODEL)
boot.to_csv(OUT / "cross_dataset_bootstrap_auc.csv", index=False)

# -----------------------------------------------------------------------------
# 6. Console summary
# -----------------------------------------------------------------------------

pd.set_option("display.width", 200)
print("\n=== External (GenImage) -- fixed in-distribution thresholds ===")
print(overall[["model", "roc_auc", "average_precision", "accuracy", "balanced_accuracy", "f1_fake",
               "threshold_source", "oracle_accuracy"]].to_string(index=False))
if source_df is not None:
    print("\n=== Generalization gap (in-distribution minus external; larger = worse transfer) ===")
    print(gap[["model", "roc_auc_in_dist", "roc_auc_external", "roc_auc_drop",
               "accuracy_in_dist", "accuracy_external", "accuracy_drop"]].to_string(index=False))
print("\n=== AUC with 95% bootstrap CI; paired difference vs", REFERENCE_MODEL, "===")
print(boot.to_string(index=False))
print("\n=== Per-generator AUC (reals + that generator's fakes) ===")
print(pd.read_csv(OUT / "cross_dataset_per_generator_auc_wide.csv").to_string(index=False))
print("\n=== Per-generator detection rate (fakes only) ===")
print(pd.read_csv(OUT / "cross_dataset_per_generator_detection_rate_wide.csv").to_string(index=False))
print(f"\nAll outputs saved to: {OUT}")